In [19]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from dotenv import load_dotenv
import os

load_dotenv()

True

In [20]:
class LLMState(TypedDict):
    review: str
    sentiments: str
    greeting: str


In [21]:
#define nodes

model = ChatOpenAI(model="gpt-4o-mini")
print(model)

def generate_llm_response(state: LLMState):
    prompt = (
        "Classify the sentiment of this review. "
        "Reply with exactly one word: Positive or Negative.\n\n"
        f"Review: {state['review']}"
    )
    return {"sentiments": model.invoke(prompt).content.strip()}

metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.5.0', 'langchain-openai': '1.8.1'}} profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True} client=<openai.resources.chat.completions.completions.Completions object at 0x000001F222613110> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F222611A90> root_client=<openai.OpenAI object at 0x000001F222613890> root_async_cli

In [22]:

def route(state: LLMState) -> Literal["positive_response", "negative_response"]:
    if "positive" in state['sentiments'].lower():
        return "positive_response"
    return "negative_response"

In [23]:
def positive_response(state: LLMState):
    prompt = f"Write a short, warm thank-you reply to this review:\n{state['review']}"
    return {"greeting": model.invoke(prompt).content}

In [24]:
def negative_response(state: LLMState):
    prompt = (
        "Write a short, empathetic apology to this review "
        f"and offer to help:\n{state['review']}"
    )
    return {"greeting": model.invoke(prompt).content}

In [25]:
graph = StateGraph(LLMState)

graph.add_node("generate_llm_response", generate_llm_response)
graph.add_node("positive_response", positive_response)
graph.add_node("negative_response", negative_response)


graph.add_edge(START, 'generate_llm_response')
graph.add_conditional_edges('generate_llm_response', route)
graph.add_edge("positive_response", END)
graph.add_edge("negative_response", END)

workflow = graph.compile()

In [26]:
print(workflow.invoke({"review": "The delivery was fast and the product works great!"}))
print(workflow.invoke({"review": "It broke after two days and support ignored me."}))


{'review': 'The delivery was fast and the product works great!', 'sentiments': 'Positive', 'greeting': "Dear [Reviewer's Name],\n\nThank you so much for your kind words! We're thrilled to hear that you were pleased with the fast delivery and that the product is working great for you. Your satisfaction means the world to us!\n\nWarm regards,  \n[Your Name/Company Name]"}
{'review': 'It broke after two days and support ignored me.', 'sentiments': 'Negative', 'greeting': "Dear [Customer's Name],\n\nI’m truly sorry to hear about your experience. It’s frustrating when something doesn’t work as expected, especially after such a short time. I apologize for our support team not getting back to you promptly; that’s certainly not the service we aim to provide.\n\nWe would love the opportunity to assist you in resolving this issue. Please reach out to us directly at [contact information], and we'll do our best to make things right.\n\nThank you for your understanding, and we hope to hear from you